# Сравнение модели: 17 фич (старая) vs 19 фич (новая)

**Цель:** Оценить влияние двух новых фич (`grazing_norm_deviation`, `natural_loss_risk_score`) на качество модели.

- **Старая модель:** 17 фич, обучена на данных без нормативов из документов МСХ
- **Новая модель:** 19 фич, те же гиперпараметры + 2 фичи на основе гос. нормативов

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("../data")
MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(exist_ok=True)

## 1. Загрузка данных

In [ ]:
df = pd.read_csv(DATA_DIR / "data_features.csv")
print(f"Датасет: {len(df):,} строк, {df.shape[1]} колонок")
print(f"Колонки: {list(df.columns)}")
df.head(3)

## 2. Определение наборов фич

In [ ]:
# Старый набор (17 фич)
OLD_FEATURES = [
    "gross_output_growth_yoy", "land_to_livestock_ratio",
    "historical_survival_rate", "subsidy_dependence_index",
    "veterinary_compliance", "years_in_operation",
    "pedigree_ratio", "previous_subsidies_count", "debt_load_ratio",
    "log_amount", "livestock_count", "direction_code",
    "is_pedigree", "is_producer", "hour_submitted",
    "month_submitted", "region_encoded",
]

# Новый набор (19 фич) — старые + 2 новые
NEW_FEATURES = OLD_FEATURES + ["grazing_norm_deviation", "natural_loss_risk_score"]

TARGET = "historical_score"

print(f"Старых фич: {len(OLD_FEATURES)}")
print(f"Новых фич: {len(NEW_FEATURES)}")
print(f"Добавлены: {[f for f in NEW_FEATURES if f not in OLD_FEATURES]}")

## 3. Загрузка старой модели (если есть)

In [ ]:
old_model_path = MODELS_DIR / "xgb_scorer.joblib"
old_scaler_path = MODELS_DIR / "scaler.joblib"

old_model_exists = old_model_path.exists()
print(f"Старая модель найдена: {old_model_exists}")

if old_model_exists:
    old_model = joblib.load(old_model_path)
    old_scaler = joblib.load(old_scaler_path)
    print(f"  Модель: {type(old_model).__name__}")
    print(f"  Деревьев: {old_model.n_estimators}")
    print(f"  Фичей в модели: {old_model.n_features_in_}")
else:
    print("  ⚠️ Старой модели нет — обучим обе с нуля для честного сравнения")

## 4. Подготовка данных (общий train/test split)

In [ ]:
# Используем ОДИН и тот же split для обоих экспериментов
X_old = df[OLD_FEATURES].copy()
y = df[TARGET].copy()

mask = X_old.notna().all(axis=1) & y.notna()
X_old = X_old[mask]
y = y[mask]

X_new = df.loc[mask, NEW_FEATURES].copy()

X_old_train, X_old_test, y_train, y_test = train_test_split(
    X_old, y, test_size=0.20, random_state=RANDOM_SEED
)

# Для новых фич — тот же индекс
X_new_train = X_new.loc[X_old_train.index]
X_new_test = X_new.loc[X_old_test.index]

print(f"Train: {len(X_old_train):,} строк")
print(f"Test:  {len(X_old_test):,} строк")
print(f"\nСтарые фичи (train): {list(X_old_train.columns)}")
print(f"Новые фичи (train):  {list(X_new_train.columns)}")

## 5. Функция обучения и оценки

In [ ]:
def build_model():
    """Создаёт XGBRegressor с теми же гиперпараметрами что в train_model.py"""
    return XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7,
        reg_lambda=1.0, reg_alpha=0.1, min_child_weight=5,
        early_stopping_rounds=50, random_state=RANDOM_SEED,
        n_jobs=-1, verbosity=0,
    )


def train_and_evaluate(X_train, X_test, y_train, y_test, label: str) -> dict:
    """Скейлит, обучает, оценивает, возвращает метрики + объекты."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    
    # Скейлинг
    scaler = StandardScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_te = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
    
    # Обучение
    model = build_model()
    model.fit(X_tr, y_train, eval_set=[(X_te, y_test)], verbose=50)
    
    # Предсказания
    y_pred_tr = np.clip(model.predict(X_tr), 1, 100)
    y_pred_te = np.clip(model.predict(X_te), 1, 100)
    
    # Метрики
    metrics = {
        "label": label,
        "n_features": X_train.shape[1],
        "n_trees": model.best_iteration + 1,
        "train_mae":  round(mean_absolute_error(y_train, y_pred_tr), 3),
        "train_rmse": round(np.sqrt(mean_squared_error(y_train, y_pred_tr)), 3),
        "train_r2":   round(r2_score(y_train, y_pred_tr), 4),
        "test_mae":   round(mean_absolute_error(y_test, y_pred_te), 3),
        "test_rmse":  round(np.sqrt(mean_squared_error(y_test, y_pred_te)), 3),
        "test_r2":    round(r2_score(y_test, y_pred_te), 4),
    }
    
    # Cross-validation (быстрая версия, 200 деревьев)
    print(f"  5-fold CV...")
    cv_model = XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0,
    )
    cv_scores = cross_val_score(cv_model, X_tr, y_train, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
    metrics["cv_mae"] = round(-cv_scores.mean(), 3)
    metrics["cv_std"] = round(cv_scores.std(), 3)
    
    # MAE по зонам
    df_err = pd.DataFrame({"y_true": y_test, "y_pred": y_pred_te})
    df_err["zone"] = pd.cut(df_err["y_true"], bins=[0, 50, 80, 100], labels=["red", "yellow", "green"])
    zone_mae = df_err.groupby("zone", observed=True).apply(
        lambda g: round(mean_absolute_error(g["y_true"], g["y_pred"]), 2)
    ).to_dict()
    metrics["mae_by_zone"] = zone_mae
    
    # Feature importance
    importances = pd.Series(model.feature_importances_, index=X_train.columns)
    metrics["feature_importance"] = importances.sort_values(ascending=False)
    
    # Печать
    print(f"  Деревьев: {metrics['n_trees']}")
    print(f"  Train MAE:  {metrics['train_mae']:.3f}  |  RMSE: {metrics['train_rmse']:.3f}  |  R²: {metrics['train_r2']:.4f}")
    print(f"  Test  MAE:  {metrics['test_mae']:.3f}  |  RMSE: {metrics['test_rmse']:.3f}  |  R²: {metrics['test_r2']:.4f}")
    print(f"  CV MAE:     {metrics['cv_mae']:.3f} ± {metrics['cv_std']:.3f}")
    print(f"  MAE по зонам: {zone_mae}")
    
    return metrics, model, scaler

## 6. Обучение обеих моделей

In [ ]:
print("🚀 Обучаю МОДЕЛЬ 1: 17 фич (старая)...")
metrics_old, model_old, scaler_old = train_and_evaluate(
    X_old_train, X_old_test, y_train, y_test, "17 фич (старая)"
)

In [ ]:
print("\n🚀 Обучаю МОДЕЛЬ 2: 19 фич (новая)...")
metrics_new, model_new, scaler_new = train_and_evaluate(
    X_new_train, X_new_test, y_train, y_test, "19 фич (новая)"
)

## 7. Сравнительная таблица метрик

In [ ]:
def compare_metrics(m_old, m_new):
    rows = []
    for key in ["train_mae", "train_rmse", "train_r2", "test_mae", "test_rmse", "test_r2", "cv_mae"]:
        old_val = m_old[key]
        new_val = m_new[key]
        delta = new_val - old_val
        pct = (delta / abs(old_val) * 100) if old_val != 0 else 0
        
        # Для MAE/RMSE меньше = лучше, для R² больше = лучше
        if "r2" in key:
            better = "✅" if delta > 0 else "❌"
        else:
            better = "✅" if delta < 0 else "❌"
        
        rows.append({
            "Метрика": key,
            "17 фич": old_val,
            "19 фич": new_val,
            "Δ": round(delta, 4),
            "Δ%": f"{pct:+.2f}%",
            "": better,
        })
    
    # Зоны
    for zone in ["red", "yellow", "green"]:
        old_z = m_old["mae_by_zone"].get(zone, None)
        new_z = m_new["mae_by_zone"].get(zone, None)
        if old_z is not None and new_z is not None:
            delta = new_z - old_z
            pct = (delta / old_z * 100) if old_z != 0 else 0
            better = "✅" if delta < 0 else "❌"
            rows.append({
                "Метрика": f"MAE зона {zone}",
                "17 фич": old_z,
                "19 фич": new_z,
                "Δ": round(delta, 4),
                "Δ%": f"{pct:+.2f}%",
                "": better,
            })
    
    return pd.DataFrame(rows)

comparison_df = compare_metrics(metrics_old, metrics_new)
display(comparison_df)

## 8. Визуализация сравнения

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MAE
metrics_names = ["Train MAE", "Test MAE", "CV MAE"]
old_vals = [metrics_old["train_mae"], metrics_old["test_mae"], metrics_old["cv_mae"]]
new_vals = [metrics_new["train_mae"], metrics_new["test_mae"], metrics_new["cv_mae"]]

x = np.arange(len(metrics_names))
w = 0.35

axes[0].bar(x - w/2, old_vals, w, label="17 фич", color="#1976d2", alpha=0.85)
axes[0].bar(x + w/2, new_vals, w, label="19 фич", color="#4caf50", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_names)
axes[0].set_ylabel("MAE (меньше = лучше)")
axes[0].set_title("MAE: Старая vs Новая")
axes[0].legend()
for i, (o, n) in enumerate(zip(old_vals, new_vals)):
    delta = n - o
    axes[0].text(i, max(o, n) + 0.1, f"{delta:+.3f}", ha="center", fontsize=10,
                 color="#4caf50" if delta < 0 else "#d32f2f", fontweight="bold")

# R²
r2_names = ["Train R²", "Test R²"]
r2_old = [metrics_old["train_r2"], metrics_old["test_r2"]]
r2_new = [metrics_new["train_r2"], metrics_new["test_r2"]]
x2 = np.arange(len(r2_names))
axes[1].bar(x2 - w/2, r2_old, w, label="17 фич", color="#1976d2", alpha=0.85)
axes[1].bar(x2 + w/2, r2_new, w, label="19 фич", color="#4caf50", alpha=0.85)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(r2_names)
axes[1].set_ylabel("R² (больше = лучше)")
axes[1].set_title("R²: Старая vs Новая")
axes[1].legend()
for i, (o, n) in enumerate(zip(r2_old, r2_new)):
    delta = n - o
    axes[1].text(i, max(o, n) + 0.005, f"{delta:+.4f}", ha="center", fontsize=10,
                 color="#4caf50" if delta > 0 else "#d32f2f", fontweight="bold")

# MAE по зонам
zones = ["red", "yellow", "green"]
zone_old = [metrics_old["mae_by_zone"].get(z, 0) for z in zones]
zone_new = [metrics_new["mae_by_zone"].get(z, 0) for z in zones]
x3 = np.arange(len(zones))
axes[2].bar(x3 - w/2, zone_old, w, label="17 фич", color="#1976d2", alpha=0.85)
axes[2].bar(x3 + w/2, zone_new, w, label="19 фич", color="#4caf50", alpha=0.85)
axes[2].set_xticks(x3)
axes[2].set_xticklabels(["🔴 Red", "🟡 Yellow", "🟢 Green"])
axes[2].set_ylabel("MAE (меньше = лучше)")
axes[2].set_title("MAE по зонам")
axes[2].legend()

plt.tight_layout()
plt.savefig(REPORTS_DIR / "compare_17_vs_19.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 Сохранён: {REPORTS_DIR / 'compare_17_vs_19.png'}")

## 9. Важность новых фич

In [ ]:
fi = metrics_new["feature_importance"].copy()
fi_norm = fi / fi.sum() * 100

fig, ax = plt.subplots(figsize=(10, 7))

colors = []
for feat in fi_norm.index:
    if feat in ["grazing_norm_deviation", "natural_loss_risk_score"]:
        colors.append("#ff6f00")  # оранжевый для новых фич
    elif fi_norm[feat] < fi_norm.median():
        colors.append("#d32f2f")
    else:
        colors.append("#1976d2")

fi_norm.plot(kind="barh", ax=ax, color=colors)
ax.set_title("Feature Importance — 19 фич (новая модель)", fontsize=14, fontweight="bold")
ax.set_xlabel("Доля важности (%)")
ax.axvline(fi_norm.median(), color="orange", linestyle="--", alpha=0.7, label="Медиана")
ax.legend()

# Подписи для новых фич
for feat in ["grazing_norm_deviation", "natural_loss_risk_score"]:
    val = fi_norm[feat]
    ax.text(val + 0.3, list(fi_norm.index).index(feat), f"{val:.1f}% ⭐",
            va="center", fontsize=10, color="#ff6f00", fontweight="bold")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "feature_importance_19.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 Сохранён: {REPORTS_DIR / 'feature_importance_19.png'}")

print("\n📊 Важность новых фич:")
for feat in ["grazing_norm_deviation", "natural_loss_risk_score"]:
    print(f"  {feat}: {fi_norm[feat]:.2f}%")

## 10. SHAP-анализ новых фич

In [ ]:
# SHAP summary plot для новой модели
explainer = shap.TreeExplainer(model_new)
shap_values = explainer.shap_values(X_new_test)

print("📊 SHAP Summary Plot (новая модель, 19 фич):")
shap.summary_plot(shap_values, X_new_test, show=False, plot_size=(10, 8))
plt.tight_layout()
plt.savefig(REPORTS_DIR / "shap_summary_19.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 Сохранён: {REPORTS_DIR / 'shap_summary_19.png'}")

## 11. Итоговый вывод

In [ ]:
print("=" * 70)
print("  ИТОГОВОЕ СРАВНЕНИЕ: 17 фич vs 19 фич")
print("=" * 70)

delta_mae = metrics_new["test_mae"] - metrics_old["test_mae"]
delta_r2 = metrics_new["test_r2"] - metrics_old["test_r2"]
delta_cv = metrics_new["cv_mae"] - metrics_old["cv_mae"]

print(f"\n  Test MAE:  {metrics_old['test_mae']:.3f} → {metrics_new['test_mae']:.3f}  ({delta_mae:+.3f}, {delta_mae/metrics_old['test_mae']*100:+.2f}%)")
print(f"  Test R²:   {metrics_old['test_r2']:.4f} → {metrics_new['test_r2']:.4f}  ({delta_r2:+.4f})")
print(f"  CV MAE:    {metrics_old['cv_mae']:.3f} → {metrics_new['cv_mae']:.3f}  ({delta_cv:+.3f}, {delta_cv/metrics_old['cv_mae']*100:+.2f}%)")

print(f"\n  Новые фичи в топ-10 по важности:")
fi = metrics_new["feature_importance"]
for i, (feat, imp) in enumerate(fi.items()):
    if feat in ["grazing_norm_deviation", "natural_loss_risk_score"]:
        print(f"    #{i+1} {feat}: {imp:.4f}")

if delta_mae < 0:
    print(f"\n  ✅ НОВАЯ МОДЕЛЬ ЛУЧШЕ: MAE улучшился на {abs(delta_mae):.3f} ({abs(delta_mae/metrics_old['test_mae']*100):.1f}%)")
elif delta_mae > 0:
    print(f"\n  ⚠️ СТАРАЯ МОДЕЛЬ ЛУЧШЕ: MAE ухудшился на {delta_mae:.3f}")
else:
    print(f"\n  ➡️ БЕЗ ИЗМЕНЕНИЙ: новые фичи не повлияли на MAE")

print("=" * 70)

## 12. Сохранение новой модели (если лучше)

In [ ]:
if metrics_new["test_mae"] <= metrics_old["test_mae"]:
    print("✅ Новая модель лучше или равна — сохраняю...")
    
    joblib.dump(model_new, MODELS_DIR / "xgb_scorer.joblib")
    joblib.dump(scaler_new, MODELS_DIR / "scaler.joblib")
    joblib.dump(explainer, MODELS_DIR / "shap_explainer.joblib")
    
    with open(MODELS_DIR / "feature_names.json", "w", encoding="utf-8") as f:
        json.dump(NEW_FEATURES, f, ensure_ascii=False, indent=2)
    
    print(f"  ✅ {MODELS_DIR}/xgb_scorer.joblib")
    print(f"  ✅ {MODELS_DIR}/scaler.joblib")
    print(f"  ✅ {MODELS_DIR}/shap_explainer.joblib")
    print(f"  ✅ {MODELS_DIR}/feature_names.json ({len(NEW_FEATURES)} фич)")
else:
    print("❌ Новая модель хуже — НЕ заменяю старую.")
    print("   Возможно, новые фичи шумят или переобучаются.")